# CobraBox Use-Case : Directed Connectivity for Localizing Epileptogenicity (NB #2: Analysis & Statistics)

Authors: *[COBRA group](https://cobra.cs.cas.cz), Institute of Computer Science, The Czech Academy of Sciences*

<div align="left">
<img src="Images/Logo_CAS_ICS.png" align="left" width="254" alt="logo ICS">
</div>


<br>
<br>

---------------------

This is the second of the two notebooks that together form a complete, reproducible example of using [CobraBox](https://github.com/cobragroup/cobrabox) on an intracranial EEG (iEEG) dataset:

1. ***Data preprocessing and connectivity estimation (notebook 2)*** turns the raw recordings of the [interictal iEEG Zurich dataset](https://openneuro.org/datasets/ds003498/versions/1.1.1) into clean, segmented, analysis-ready data, computes directed connectivity using directed transfer function (DTF), and calculates the inward and outward strength of each network node (electrode).
2. ***Analysis and statistics (this notebook)*** runs the analysis and statistics to localize the epileptogenic tissue and predict the surgical outcome for the patients in the cohort.

Both notebooks follow the pipeline of the accompanying study (Stergiadis C, Halliday DM, Kazis D, Klados MA. *High-frequency directed networks can identify epileptogenic tissue and predict surgical outcome in drug-resistant epilepsy*. Epilepsy Research. 2026;226:107838. doi: [10.1016/j.eplepsyres.2026.107838](https://doi.org/10.1016/j.eplepsyres.2026.107838)).

<span style="color:darkred">
IMPORTANT: As in notebook 2, the routine steps that are <b>not</b> the point of this use-case (loading the patient metadata, loading the saved segments, and reading/writing intermediate files) are handled by the local helper module <code>local_utils.py</code> and called explicitly as <code>local_utils.&lt;function&gt;()</code>. The analysis itself (inward/outward strength, the statistics and the figure) is written out in full in the notebook. Note the relative import <code>import local_utils</code> (not <code>from local_utils import *</code>).
</span>

***IMPORTANT***: For this notebook to run, you first ***must*** run 02_preprocessing_and_connectivity estimation.ipynb

### Contents of this notebook

1. Median strengths inside and outside the surgical resection
2. Statistical comparison inside vs. outside the resection
3. Classification of epileptogenic tissue
4. Surgical outcome prediction
5. Where we end up …

### Import dependencies

Running this notebook requires ***python*** (>=3.11), ***xarray*** (>2026.2.0) and ***cobrabox***
(>=X.Y). The analysis and figures additionally use ***NumPy***, ***pandas***, ***SciPy***,
***scikit-learn*** (for the logistic-regression classifier and cross-validation) and
***Matplotlib***. The routine helpers live in the local module ***local_utils.py*** that ships
alongside these notebooks.

In [ ]:
## IMPORT THE PACKAGES NEEDED TO RUN THE NOTEBOOK
# Python standard library imports
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

%matplotlib inline
from scipy.stats import wilcoxon, fisher_exact

# scikit-learn: logistic-regression classifier, cross-validation and ROC/AUC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve

import cobrabox as cb

# Local libraries and modules
import local_utils

### Setup: configuration and the nodal-strength table

This notebook depends only on the **files that notebook 2 wrote to disk** (the band-averaged DTF
connectivity matrices), not on notebook 2's live variables. To stay self-contained, we therefore
first re-declare the same configuration as notebook 2 and then rebuild the tidy nodal-strength table
`strength_df` directly from the saved connectivity, using the exact same definitions of inward and
outward strength. If you have just run notebook 2 in the same session this simply reproduces the
`strength_df` you already have; if you are starting a fresh kernel it reconstructs it from disk.

`strength_df` has one row per **(subject, band, contact)** with the min-max–normalised `in_strength`
and `out_strength`, the surgical `outcome` group, and whether the contact lies `inside` or `outside`
the resection.

In [ ]:
# --- Configuration (mirrors notebook 2 so this notebook can run on its own) ---
cb.set_dataset_dir(Path(".") / "data", persist=False)

SUBJECTS = [f"sub-{i:02d}" for i in range(1, 21)]
CONNECTIVITY_METHOD = "dtf"           # label used in the saved connectivity file names
BANDS = local_utils.BANDS             # eight frequency bands, identical to notebook 2
band_names = list(BANDS)

# Patient metadata: outcome (ILAE grade) and the list of resected bipolar pairs
patient_info = local_utils.load_patient_info()


def outcome_group(subject_id):
    """Good outcome = ILAE 1, poor outcome = ILAE 2-6 (adjust to your metadata format)."""
    grade = str(patient_info[subject_id]["outcome"]).upper().replace(" ", "")
    return "good" if grade in ("ILAE1", "1") else "poor"


# --- Inward/outward strength (same definitions as notebook 2) ---
def inward_outward(matrix):
    """Inward and outward strength from one (space_to x space_from) matrix.

    Entry [i, j] is the influence from channel j to channel i.
    inward[i]  = mean over j of [i, j]   (influence received)
    outward[j] = mean over i of [i, j]   (influence sent)
    """
    m = matrix.transpose("space_to", "space_from").values.astype(float).copy()
    np.fill_diagonal(m, np.nan)  # ignore self-connections
    inward = np.nanmean(m, axis=1)
    outward = np.nanmean(m, axis=0)
    return inward, outward


def minmax(x):
    """Rescale a vector to [0, 1]; a flat vector becomes all zeros."""
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


# --- Rebuild the tidy strength table from the connectivity saved by notebook 2 ---
rows = []
for subject_id in SUBJECTS:
    conn = local_utils.load_connectivity(subject_id, method=CONNECTIVITY_METHOD)
    resected = set(patient_info[subject_id]["resected"])
    grp = outcome_group(subject_id)

    for band, matrix in conn.items():
        inward, outward = inward_outward(matrix)
        inward, outward = minmax(inward), minmax(outward)
        channels = matrix.coords["space_to"].values
        for ch, in_s, out_s in zip(channels, inward, outward):
            rows.append(
                {
                    "subject": subject_id,
                    "outcome": grp,
                    "band": band,
                    "channel": ch,
                    "region": "inside" if ch in resected else "outside",
                    "in_strength": float(in_s),
                    "out_strength": float(out_s),
                }
            )

strength_df = pd.DataFrame(rows)
print(f"{len(strength_df)} rows  ({strength_df['subject'].nunique()} subjects, {len(band_names)} bands)")
strength_df.head()

### What this notebook computes

Using the connectivity saved by notebook 2 (`02_preprocessing_final_christos.ipynb`), we reproduce
and extend the analysis of (Stergiadis et al., 2026). For every contact we have how strongly it
**sends** influence to the rest of the network (**outward strength**) and how strongly it **receives**
influence (**inward strength**). Starting from these two numbers per contact and band, the notebook
does four things:

1. **Localisation (sections 1-2).** Compare inward/outward strength **inside** vs. **outside** the
   resection, separately for **good-** and **poor-**outcome patients, with a paired statistical test
   and a figure.
2. **Classification (section 3).** Ask whether the two strength values alone can *identify*
   epileptogenic (resected) tissue, by training a logistic-regression classifier with balanced
   resampling and cross-validation, following the methodology of (Stergiadis et al., 2026).
3. **Outcome prediction (section 4).** Ask whether the same strength values can *predict the surgical
   outcome*, using a simple per-patient contrast between resected and spared tissue and
   leave-one-out cross-validation.

> **What the study found (Stergiadis et al., 2026, §3.2).** In good-outcome patients, resected tissue
> showed **lower outward strength** than the rest of the brain in HFO-free segments, most clearly at the higher frequency
> bands; inward strength showed a similar but non-significant trend, and poor-outcome patients showed no such
> difference. This fits the idea that epileptogenic tissue is *functionally isolated* by the surrounding network
> during the quiet period between seizures (interictal period).

## 1. Median strengths inside vs outside the resection

We now summarise each patient with two values per band: the **median** strength of
the contacts **inside** the resection and the **median** of those **outside**. The median (rather than
the mean) is used because some patients have only a few electrodes, where a single outlier could
easily distort the average.

In [ ]:
# Long -> per-patient medians, one row per (subject, band, measure) with inside/outside columns
long = strength_df.melt(
    id_vars=["subject", "outcome", "band", "region"],
    value_vars=["in_strength", "out_strength"],
    var_name="measure",
    value_name="value",
)

medians = (
    long.groupby(["subject", "outcome", "band", "measure", "region"])["value"]
    .median()
    .reset_index()
    .pivot_table(index=["subject", "outcome", "band", "measure"], columns="region", values="value")
    .reset_index()
)
medians.columns.name = None
for col in ("inside", "outside"):
    if col not in medians.columns:
        medians[col] = np.nan

medians.head(8)

## 2. Statistical testing

Within each outcome group (good / poor), for each band and each measure, we compare the per-patient
**inside** and **outside** medians with a **paired Wilcoxon signed-rank test** (a non-parametric test
suited to small, paired samples). Because we test eight bands at once, we correct the p-values with the
**Benjamini–Hochberg** false-discovery-rate procedure. A result is called significant when the
corrected value `q < 0.05`.

In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR correction; NaNs are preserved."""
    p = np.asarray(pvals, dtype=float)
    mask = ~np.isnan(p)
    q = np.full_like(p, np.nan)
    pv = p[mask]
    m = pv.size
    if m:
        order = np.argsort(pv)
        adj = pv[order] * m / (np.arange(m) + 1)
        adj = np.minimum.accumulate(adj[::-1])[::-1]
        out = np.empty(m)
        out[order] = np.clip(adj, 0, 1)
        q[mask] = out
    return q


band_names = list(BANDS)
results = []
for outcome in ("good", "poor"):
    for measure in ("out_strength", "in_strength"):
        recs = []
        sub = medians[(medians["outcome"] == outcome) & (medians["measure"] == measure)]
        for band in band_names:
            b = sub[sub["band"] == band].dropna(subset=["inside", "outside"])
            if len(b) < 2:
                p = np.nan
            else:
                try:
                    _, p = wilcoxon(b["inside"].values, b["outside"].values)
                except ValueError:
                    p = np.nan
            recs.append(
                {
                    "outcome": outcome,
                    "measure": measure,
                    "band": band,
                    "n_patients": len(b),
                    "p_value": p,
                }
            )
        for r, q in zip(recs, bh_fdr([r["p_value"] for r in recs])):
            r["q_value"] = q
            r["significant"] = bool(q < 0.05) if not np.isnan(q) else False
            results.append(r)

stats_df = pd.DataFrame(results)
pd.set_option("display.float_format", "{:.4f}".format)
print(stats_df.to_string(index=False))

### Visualising the inside-vs-outside comparison

For each measure we plot, per band, the per-patient **inside** (orange) and **outside** (blue)
medians, with a **Good** and a **Poor** block. Grey lines link the two values of the same patient, and
a `*` marks bands that survive the FDR correction.

In [ ]:
def strength_plot(measure, measure_label):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for ax, band in zip(axes.flatten(), band_names):
        for grp, x0 in (("good", 0.0), ("poor", 2.6)):
            b = medians[
                (medians["outcome"] == grp)
                & (medians["measure"] == measure)
                & (medians["band"] == band)
            ].dropna(subset=["inside", "outside"])
            inside, outside = b["inside"].values, b["outside"].values
            xin, xout = x0, x0 + 1.0
            for a, c in zip(inside, outside):
                ax.plot([xin, xout], [a, c], color="gray", alpha=0.4, lw=0.7)
            ax.scatter(np.full(len(inside), xin), inside, color="tomato", s=18, zorder=3)
            ax.scatter(np.full(len(outside), xout), outside, color="steelblue", s=18, zorder=3)
            if len(inside):
                ax.plot([xin - 0.18, xin + 0.18], [np.median(inside)] * 2, color="black", lw=2)
            if len(outside):
                ax.plot([xout - 0.18, xout + 0.18], [np.median(outside)] * 2, color="black", lw=2)
            row = stats_df[
                (stats_df["outcome"] == grp)
                & (stats_df["measure"] == measure)
                & (stats_df["band"] == band)
            ]
            if len(row) and bool(row["significant"].values[0]):
                ax.text((xin + xout) / 2, 1.04, "*", ha="center", va="bottom", fontsize=15)
        ax.set_title(band, fontsize=9)
        ax.set_xticks([0.5, 3.1])
        ax.set_xticklabels(["Good", "Poor"], fontsize=9)
        ax.set_ylim(-0.05, 1.15)
        ax.set_ylabel(measure_label, fontsize=8)

    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tomato",
            markersize=8,
            label="inside resection",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="steelblue",
            markersize=8,
            label="outside resection",
        ),
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02)
    )
    fig.suptitle(
        f"{CONNECTIVITY_METHOD.upper()} {measure_label}: inside vs outside resection",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


strength_plot("out_strength", "outward strength")  # headline result
strength_plot("in_strength", "inward strength")  # shown for comparison

## 3. Classification of epileptogenic tissue

Sections 1-2 asked whether resected tissue *differs* from the rest of the brain. Here we ask the
sharper, decision-oriented question: given only a contact's **inward and outward strength**, can we
**classify it as epileptogenic (resected) or not**? This mirrors the classifier analysis of
(Stergiadis et al., 2026): the two strengths are used as features to train a **logistic-regression
(LR)** classifier, one model per frequency band.

The difficulty is that the two classes are **very imbalanced** — far fewer resected contacts than
spared ones — so a classifier could look accurate simply by labelling everything "non-resected".
Following the paper's methodology, we handle this with **balanced resampling repeated many times**:

1. Keep **all** resected contacts (the minority class, *Class 1*).
2. Draw a **random equal-sized subset** of non-resected contacts (*Class 2*).
3. On this balanced subset, run **5-fold cross-validation**, pool the held-out (out-of-fold)
   predictions into a single ROC curve, and record its **AUC**.
4. Repeat steps 2-3 for **100** independent random subsets and report the **mean AUC** (± s.d.),
   which is robust to the particular non-resected contacts that happen to be drawn.

We show the result as a **cross-validated ROC curve per band** — the mean over the 100 subsets with a shaded ±1 SD band — followed by a bar-chart summary of the per-band AUCs.

**A note on the patient set.** A resected contact is only a reliable proxy for *epileptogenic* tissue
when the surgery actually worked, i.e. in **good-outcome** patients; in poor-outcome patients the
resected label is noisier. We therefore train on the **good-outcome patients** by default
(`CLASSIFIER_SUBJECTS = [s for s in SUBJECTS if outcome_group(s) == "good"]`). Uncomment the second
line to fall back to pooling all patients.

In [ ]:
# # I'm putting here the code from another version, to think 

# # ---- Per-patient contrast feature: delta = median(inside) - median(outside) ----
# # `medians` was built in section 1 (one row per subject x band x measure, with inside/outside columns).
# medians_delta = medians.copy()
# medians_delta["delta"] = medians_delta["inside"] - medians_delta["outside"]

# N_SUBSETS_OUTCOME = 50   # balanced subsets per band/measure
# N_FOLDS_OUTCOME = 5       # k-fold within each subset
# N_PERM_OUTCOME = 50      # permutations for p-value
# BASE_SEED_OUTCOME = 99


# def balanced_auc(feature, y, n_subsets=N_SUBSETS_OUTCOME, n_folds=N_FOLDS_OUTCOME, seed=0):
#     """Mean cross-validated AUC over balanced random subsets.

#     Mirrors the section-3 approach: each iteration samples all minority-class patients
#     and an equal random draw of majority-class patients, then runs stratified k-fold CV.
#     Returns (array of per-subset AUCs, n_patients_used).
#     """
#     feature = np.asarray(feature, float)
#     y = np.asarray(y, int)
#     ok = ~np.isnan(feature)
#     feature, y = feature[ok], y[ok]
#     if len(np.unique(y)) < 2 or len(y) < 4:
#         return np.array([np.nan]), int(len(y))

#     pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
#     maj, minn = (pos, neg) if len(pos) > len(neg) else (neg, pos)
#     n_min = len(minn)
#     if n_min < n_folds:
#         return np.array([np.nan]), int(len(y))

#     rng = np.random.default_rng(seed)
#     aucs = []
#     for rep in range(n_subsets):
#         sel = rng.choice(maj, size=n_min, replace=False)
#         idx = np.concatenate([minn, sel])
#         Xi, yi = feature[idx].reshape(-1, 1), y[idx]
#         clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
#         cv = StratifiedKFold(n_folds, shuffle=True, random_state=rep)
#         proba = cross_val_predict(clf, Xi, yi, cv=cv, method="predict_proba")[:, 1]
#         aucs.append(roc_auc_score(yi, proba))
#     return np.array(aucs), int(len(y))


# def perm_pvalue(feature, y, obs_auc, n_perm=N_PERM_OUTCOME, seed=0):
#     """Permutation p-value: fraction of null mean-AUCs >= observed."""
#     rng = np.random.default_rng(seed + 10_000)
#     null = []
#     for _ in range(n_perm):
#         aucs, _ = balanced_auc(feature, rng.permutation(y), seed=int(rng.integers(1_000_000)))
#         m = np.nanmean(aucs)
#         if not np.isnan(m):
#             null.append(m)
#     return float(np.mean(np.array(null) >= obs_auc))


# pred_rows = []
# for measure in ("out_strength", "in_strength"):
#     print(measure)
#     for b, band in enumerate(band_names):
#         print(f"  {band}")
#         m = medians_delta[(medians_delta["band"] == band) & (medians_delta["measure"] == measure)]
#         feature = m["delta"].values
#         y = (m["outcome"].values == "good").astype(int)
#         aucs, n = balanced_auc(feature, y, seed=BASE_SEED_OUTCOME + b)
#         mean_auc = float(np.nanmean(aucs))
#         std_auc = float(np.nanstd(aucs))
#         p = perm_pvalue(feature, y, mean_auc, seed=BASE_SEED_OUTCOME + b)
#         pred_rows.append({"measure": measure, "band": band,
#                           "mean_auc": mean_auc, "std_auc": std_auc,
#                           "n_patients": n, "p_value": p})

# pred_df = pd.DataFrame(pred_rows)
# print(f"Outcome prediction: balanced subsampling "
#       f"({N_SUBSETS_OUTCOME} subsets, {N_FOLDS_OUTCOME}-fold CV, {N_PERM_OUTCOME} permutations)")
# print()
# auc_table = pred_df.pivot(index="band", columns="measure", values="mean_auc").reindex(band_names)
# p_table   = pred_df.pivot(index="band", columns="measure", values="p_value").reindex(band_names)
# print("Mean AUC:")
# print(auc_table.round(4).to_string())
# print("\nPermutation p-value:")
# print(p_table.round(3).to_string())

In [ ]:
# # Code from another version, maybe will be useful

# fig, ax = plt.subplots(figsize=(9, 4.5))
# x = np.arange(len(band_names))
# w = 0.38

# for k, (measure, colour, lbl) in enumerate([
#     ("out_strength", "steelblue", "outward strength"),
#     ("in_strength",  "tomato",    "inward strength"),
# ]):
#     sub = pred_df[pred_df["measure"] == measure].set_index("band").reindex(band_names)
#     vals  = sub["mean_auc"].values
#     errs  = sub["std_auc"].values
#     pvals = sub["p_value"].values
#     ax.bar(x + (k - 0.5) * w, vals, width=w, yerr=errs, capsize=3,
#            color=colour, alpha=0.9, label=lbl)
#     for xi, v, e, p in zip(x + (k - 0.5) * w, vals, errs, pvals):
#         if not np.isnan(p) and p < 0.05:
#             ax.text(xi, v + e + 0.02, "*", ha="center", va="bottom", fontsize=12)

# ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance (AUC = 0.5)")
# ax.set_xticks(x)
# ax.set_xticklabels(band_names, rotation=30, ha="right", fontsize=9)
# ax.set_ylabel("mean AUC ± s.d.")
# ax.set_ylim(0.0, 1.0)
# ax.set_title(
#     "Predicting surgical outcome from the resected-vs-spared strength contrast\n"
#     "(* = permutation p < 0.05)"
# )
# ax.legend(fontsize=9)
# plt.tight_layout()
# plt.show()

In [ ]:
# ---- Balanced-subsampling logistic-regression classifier (Stergiadis et al., 2026) ----
N_SUBSETS = 100     # number of balanced random subsets
N_FOLDS = 5         # k for the k-fold cross-validation
BASE_SEED = 0       # for reproducibility
MEAN_FPR = np.linspace(0, 1, 200)   # common grid for averaging ROC curves across subsets

# Which patients define "epileptogenic" (resected) tissue. A resected contact is only a
# reliable proxy for epileptogenic tissue when surgery succeeded, so we train on GOOD-outcome
# patients by default; uncomment the second line to pool all patients (paper's methods text).
CLASSIFIER_SUBJECTS = [s for s in SUBJECTS if outcome_group(s) == "good"]
# CLASSIFIER_SUBJECTS = SUBJECTS

clf_data = strength_df[strength_df["subject"].isin(CLASSIFIER_SUBJECTS)]


def classify_band(df_band, seed):
    """Cross-validated ROC over N_SUBSETS balanced subsets, for one band.

    Features: inward and outward strength. Label: 1 = inside resection, 0 = outside.
    Each subset keeps all resected contacts and an equal random sample of non-resected
    ones; a 5-fold CV then yields out-of-fold probabilities from which one ROC/AUC is
    computed. Returns the array of AUCs and the per-subset TPRs interpolated onto MEAN_FPR.
    """
    rng = np.random.default_rng(seed)
    X = df_band[["in_strength", "out_strength"]].to_numpy(float)
    y = (df_band["region"].values == "inside").astype(int)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n = pos.size
    if n < N_FOLDS or neg.size < n:          # too few resected contacts to run k-fold CV
        return np.array([np.nan]), None

    aucs, tprs = [], []
    for rep in range(N_SUBSETS):
        sel = rng.choice(neg, size=n, replace=False)     # balance the two classes
        idx = np.concatenate([pos, sel])
        Xi, yi = X[idx], y[idx]
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=rep)
        proba = cross_val_predict(clf, Xi, yi, cv=cv, method="predict_proba")[:, 1]
        aucs.append(roc_auc_score(yi, proba))
        fpr, tpr, _ = roc_curve(yi, proba)
        interp = np.interp(MEAN_FPR, fpr, tpr)
        interp[0] = 0.0
        tprs.append(interp)
    return np.array(aucs), np.array(tprs)


clf_rows, roc_curves = [], {}
for b, band in enumerate(band_names):
    aucs, tprs = classify_band(clf_data[clf_data["band"] == band], seed=BASE_SEED + b)
    clf_rows.append(
        {"band": band, "mean_auc": np.nanmean(aucs), "std_auc": np.nanstd(aucs),
         "n_resected": int((clf_data[clf_data["band"] == band]["region"] == "inside").sum())}
    )
    if tprs is None:
        roc_curves[band] = None
    else:
        mean_tpr = tprs.mean(axis=0); mean_tpr[-1] = 1.0
        roc_curves[band] = (mean_tpr, tprs.std(axis=0), np.nanmean(aucs), np.nanstd(aucs))

clf_df = pd.DataFrame(clf_rows)

# Shared styling for the section's figures (cohesive, publication-style look)
PRO_STYLE = {
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "0.35", "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "0.90", "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "xtick.color": "0.35", "ytick.color": "0.35",
    "xtick.labelcolor": "0.15", "ytick.labelcolor": "0.15",
    "legend.frameon": False,
}
BAND_COLORS = dict(zip(band_names, plt.cm.viridis(np.linspace(0.12, 0.88, len(band_names)))))

pd.set_option("display.float_format", "{:.4f}".format)
print(f"Patients used: {len(CLASSIFIER_SUBJECTS)}  |  {N_SUBSETS} balanced subsets  |  {N_FOLDS}-fold CV")
print(clf_df.to_string(index=False))

In [ ]:
# ---- Figure: cross-validated ROC per band (mean of the 100 balanced subsets, shaded +/-1 SD) ----
with plt.rc_context(PRO_STYLE):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.6), sharex=True, sharey=True)
    for ax, band in zip(axes.flat, band_names):
        ax.plot([0, 1], [0, 1], ls=(0, (4, 4)), lw=1.1, color="0.65", zorder=1)  # chance
        rc = roc_curves[band]
        if rc is None:
            ax.text(0.5, 0.5, "insufficient\nresected contacts",
                    ha="center", va="center", fontsize=9.5, color="0.5")
        else:
            mean_tpr, std_tpr, mauc, sauc = rc
            colour = BAND_COLORS[band]
            ax.fill_between(MEAN_FPR, np.clip(mean_tpr - std_tpr, 0, 1),
                            np.clip(mean_tpr + std_tpr, 0, 1),
                            color=colour, alpha=0.20, linewidth=0, zorder=2)
            ax.plot(MEAN_FPR, mean_tpr, color=colour, lw=2.4, solid_capstyle="round", zorder=3)
            ax.text(0.955, 0.06, rf"AUC = {mauc:.2f} $\pm$ {sauc:.2f}",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=9.5, color="0.15",
                    bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="0.82", alpha=0.92))
        ax.set_title(band, fontweight="semibold", color="0.15")
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xticks([0, 0.5, 1.0]); ax.set_yticks([0, 0.5, 1.0])
    fig.supxlabel("False positive rate", fontsize=12)
    fig.supylabel("True positive rate", fontsize=12)
    fig.suptitle(
        "Cross-validated ROC by frequency band \u2014 resected vs. non-resected contacts\n"
        rf"mean of {N_SUBSETS} balanced subsets ($\pm$1 SD), "
        f"{len(CLASSIFIER_SUBJECTS)} good-outcome patients",
        fontsize=13.5, fontweight="semibold",
    )
    fig.tight_layout()
    plt.show()

In [ ]:
# ---- Figure: mean cross-validated AUC per band (summary of the ROC curves above) ----
with plt.rc_context(PRO_STYLE):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    x = np.arange(len(band_names))
    colours = [BAND_COLORS[b] for b in band_names]
    ax.bar(x, clf_df["mean_auc"], yerr=clf_df["std_auc"], capsize=3.5,
           color=colours, edgecolor="white", linewidth=0.8,
           error_kw=dict(ecolor="0.35", lw=1.1), zorder=3)
    ax.axhline(0.5, color="0.55", ls=(0, (4, 4)), lw=1.1, zorder=1)
    ax.text(len(band_names) - 0.45, 0.508, "chance", fontsize=9, color="0.45", va="bottom", ha="right")
    for xi, (m, s) in enumerate(zip(clf_df["mean_auc"], clf_df["std_auc"])):
        ax.text(xi, min(m + s + 0.015, 1.02), f"{m:.2f}", ha="center", va="bottom",
                fontsize=9, color="0.2")
    ax.set_xticks(x); ax.set_xticklabels(band_names, rotation=25, ha="right")
    ax.set_ylabel("cross-validated AUC")
    ax.set_ylim(0.45, 1.06)
    ax.set_title("Classifier performance by frequency band", fontweight="semibold", color="0.15", pad=12)
    ax.margins(x=0.02)
    fig.tight_layout()
    plt.show()

## 4. Surgical outcome prediction

The final question is whether the same nodal-strength information can **predict the surgical
outcome** at the *patient* level (good vs. poor). The reasoning follows a now-standard line in the
iEEG-network literature: if the resection removed the tissue that is *distinctive* in the connectivity
network, the patient tends to become seizure-free; if it did not, seizures tend to persist. Several
groups have shown that the **overlap between the resection and the network-distinctive nodes predicts
outcome**:

- Shah et al. (2019) found that greater overlap between resected contacts and the most strongly
  connected interictal contacts was associated with favourable outcome, and that good-outcome patients
  had higher connectivity localised *within* the resection zone (Shah P, Bernabei JM, Kini LG, et al.
  *High interictal connectivity within the resection zone is associated with favorable post-surgical
  outcomes in focal epilepsy patients.* NeuroImage: Clinical. 2019;23:101908.
  doi:[10.1016/j.nicl.2019.101908](https://doi.org/10.1016/j.nicl.2019.101908)).
- Sinha et al. (2017) predicted ILAE outcome from node-level iEEG features using **leave-one-out
  cross-validation** on a comparable small cohort (Sinha N, Dauwels J, Kaiser M, et al. *Predicting
  neurosurgical outcomes in focal epilepsy patients using computational modelling.* Brain.
  2017;140(2):319-332. doi:[10.1093/brain/aww299](https://doi.org/10.1093/brain/aww299)).
- Corona et al. (2023) likewise showed that mapping epileptogenic networks predicts surgical outcome
  (Corona L, Tamilia E, Perry MS, et al. *Non-invasive mapping of epileptogenic networks predicts
  surgical outcome.* Brain. 2023;146(5):1916-1931.
  doi:[10.1093/brain/awac477](https://doi.org/10.1093/brain/awac477)).

We give **two complementary approaches**: a compact contrast-plus-cross-validation baseline
(**4.1**), and the **electrode-overlap procedure of the accompanying study** (**4.2**, Stergiadis et
al., 2026), which turns the classifier of section 3 into a per-patient overlap score and tests it with
Fisher's exact test.

> **Caveat.** With ~20 patients these results are exploratory and have wide confidence intervals; they
> illustrate the pipeline rather than establish a validated predictor.

### 4.1 Simple resected-vs-spared contrasts

For each patient and band we summarise the resection with a **single contrast between resected and
spared tissue**, computed from the per-patient medians of section 1, in two complementary forms:

- an **additive** contrast, `delta = median(strength inside) - median(strength outside)`;
- a **multiplicative** contrast, `ratio = median(strength inside) / median(strength outside)`.

Sections 1-2 showed the resected-vs-spared difference is negative for outward strength in good-outcome
patients (resected tissue is functionally isolated) and near zero in poor-outcome patients — so both a
difference at/below zero and a ratio below one flag "the resection removed the network-distinctive
tissue". We feed each contrast, per measure and band, to a logistic-regression model evaluated by
**leave-one-out cross-validation** (appropriate for the small cohort, n = 20) and report the AUC. The
ratio is undefined when a patient's spared-tissue median is zero; those patients are dropped for the
ratio only.

In [ ]:
# ---- Per-patient contrasts: difference and ratio of resected vs. spared median strength ----
# `medians` (section 1) has one row per subject x band x measure, with inside/outside columns.
medians_c = medians.copy()
medians_c["delta"] = medians_c["inside"] - medians_c["outside"]              # additive contrast
with np.errstate(divide="ignore", invalid="ignore"):
    medians_c["ratio"] = medians_c["inside"] / medians_c["outside"]          # multiplicative contrast
medians_c["ratio"] = medians_c["ratio"].replace([np.inf, -np.inf], np.nan)   # spared median == 0 -> undefined


def loo_auc(feature, y):
    """Leave-one-out cross-validated ROC-AUC for a single-feature LR classifier."""
    feature = np.asarray(feature, float)
    y = np.asarray(y, int)
    ok = ~np.isnan(feature)
    feature, y = feature[ok], y[ok]
    if len(np.unique(y)) < 2 or len(y) < 4:
        return np.nan, int(len(y))
    proba = np.zeros(len(y))
    for tr, te in LeaveOneOut().split(feature):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
        clf.fit(feature[tr].reshape(-1, 1), y[tr])
        proba[te] = clf.predict_proba(feature[te].reshape(-1, 1))[:, 1]
    return roc_auc_score(y, proba), int(len(y))


pred_rows = []
for measure in ("out_strength", "in_strength"):
    for contrast in ("delta", "ratio"):
        for band in band_names:
            m = medians_c[(medians_c["band"] == band) & (medians_c["measure"] == measure)]
            y = (m["outcome"].values == "good").astype(int)
            auc, n = loo_auc(m[contrast].values, y)
            pred_rows.append({"measure": measure, "contrast": contrast,
                              "band": band, "auc": auc, "n_patients": n})

pred_df = pd.DataFrame(pred_rows)
pd.set_option("display.float_format", "{:.4f}".format)
print("Leave-one-out outcome prediction (good vs. poor) from the resected-vs-spared contrasts:")
print(pred_df.pivot(index="band", columns=["measure", "contrast"], values="auc")
             .reindex(band_names).to_string())


In [ ]:
# ---- Figure: LOOCV outcome-prediction AUC per band, difference vs. ratio contrast ----
_measures = [("out_strength", "#2C6E9E", "outward strength"),
             ("in_strength", "#C0453A", "inward strength")]
_contrasts = [("delta", "\u0394 = inside \u2212 outside"),
              ("ratio", "ratio = inside / outside")]
with plt.rc_context(PRO_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    x = np.arange(len(band_names))
    w = 0.38
    for ax, (contrast, ctitle) in zip(axes, _contrasts):
        for k, (measure, colour, mlbl) in enumerate(_measures):
            vals = (pred_df[(pred_df["measure"] == measure) & (pred_df["contrast"] == contrast)]
                    .set_index("band").reindex(band_names)["auc"].values)
            ax.bar(x + (k - 0.5) * w, vals, width=w, color=colour, alpha=0.92,
                   edgecolor="white", linewidth=0.7, label=mlbl, zorder=3)
        ax.axhline(0.5, color="0.55", ls=(0, (4, 4)), lw=1.1, zorder=1)
        ax.set_xticks(x)
        ax.set_xticklabels(band_names, rotation=25, ha="right")
        ax.set_ylim(0.0, 1.0)
        ax.set_title(ctitle, fontweight="semibold", color="0.15")
        ax.margins(x=0.02)
    axes[0].set_ylabel("leave-one-out AUC")
    axes[0].text(len(band_names) - 0.5, 0.515, "chance", fontsize=9, color="0.45",
                 va="bottom", ha="right")
    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="lower center", ncol=2, frameon=False, fontsize=10.5,
               bbox_to_anchor=(0.5, -0.03), title="strength measure")
    fig.suptitle("Predicting surgical outcome from the resected-vs-spared strength contrast",
                 fontsize=13, fontweight="semibold")
    fig.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()


### 4.2 Resected-node overlap and Youden-optimal threshold (Stergiadis et al., 2026)

This is the procedure of the accompanying study (Stergiadis et al., 2026). It reuses the section-3
classifier, but restricted to the **one graph property that was significant inside vs. outside the
resection in good-outcome patients** (section 2) — here **outward strength** — and turns it into a
patient-level prediction in four steps:

1. **Node threshold (Youden).** On the **good-outcome** contacts, fit the single-feature classifier
   (resected vs. non-resected) and take the probability threshold that maximises the **Youden index**
   *J = sensitivity + specificity − 1*, i.e. the point that best separates the two classes on the ROC
   curve. This threshold defines the epileptogenic nodes: **"hubs"** if the property is *higher* inside
   the resection, or **"sinks"** if it is *lower*. Because outward strength is lower inside the
   resection, the epileptogenic nodes here are **sinks**.
2. **Electrode overlap.** Apply that classifier+threshold to **every patient's** contacts and, for each
   patient, compute the *electrode overlap* = (number of resected contacts flagged as hubs/sinks) /
   (total number of resected contacts).
3. **Overlap-threshold sweep.** Sweep an overlap threshold from **0 % to 100 % in 10 % steps**. At each
   step, label a patient "predicted good" if their overlap *exceeds* the threshold, and tabulate
   TP (above & good), FP (above & poor), TN (below & poor), FN (below & good), from which we get the
   **PPV**, **NPV** and **accuracy**. The optimal overlap threshold is the one with the largest Youden
   index across the sweep.
4. **Fisher's exact test.** At the optimal overlap threshold, form the 2×2 table
   `[[TP, FP], [FN, TN]]` and run **Fisher's exact test**: a significant result means that resecting
   hubs/sinks above that overlap threshold is associated with good outcome.

The property and its hub/sink direction are read automatically from the section-2 results, so the cell
adapts if a different measure or band turns out to be the significant one.

The results are summarised as a **radar plot across the eight bands**, showing PPV, NPV and accuracy at each band's optimal overlap threshold; band labels carry a `*` where Fisher's exact test is significant (p < 0.05).

In [ ]:
# ---- Electrode-overlap outcome prediction (Stergiadis et al., 2026) ----
# The graph property to use = the one significant inside-vs-outside in GOOD-outcome patients (section 2).
_sig_good = stats_df[(stats_df["outcome"] == "good") & (stats_df["significant"])]
OUTCOME_MEASURE = (_sig_good["measure"].value_counts().idxmax()
                   if len(_sig_good) else "out_strength")
SIG_BANDS = set(_sig_good.loc[_sig_good["measure"] == OUTCOME_MEASURE, "band"])
OVERLAP_STEPS = np.round(np.arange(0.0, 1.0001, 0.10), 2)   # 0%, 10%, ..., 100%

print(f"Significant good-outcome property (section 2): {OUTCOME_MEASURE}")
print(f"Significant bands: {sorted(SIG_BANDS, key=band_names.index) or '(none - showing all bands)'}\n")


def flag_epileptogenic(measure, band):
    """Fit a single-feature LR on GOOD-outcome contacts, take the Youden-optimal probability
    threshold, and flag every contact (all patients) as an epileptogenic node ('hub'/'sink')."""
    df = strength_df[strength_df["band"] == band].copy()
    good = df["outcome"] == "good"
    Xg = df.loc[good, [measure]].to_numpy(float)
    yg = (df.loc[good, "region"] == "inside").astype(int).to_numpy()

    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    clf.fit(Xg, yg)
    fpr, tpr, thr = roc_curve(yg, clf.predict_proba(Xg)[:, 1])
    p_thr = float(thr[np.argmax(tpr - fpr)])                 # Youden-optimal probability threshold

    med_in = df.loc[good & (df["region"] == "inside"), measure].median()
    med_out = df.loc[good & (df["region"] == "outside"), measure].median()
    direction = "hub" if med_in > med_out else "sink"        # higher inside -> hub, lower inside -> sink

    df["is_node"] = clf.predict_proba(df[[measure]].to_numpy(float))[:, 1] >= p_thr
    return df, p_thr, direction


def patient_overlap(df):
    """Per-patient overlap = fraction of resected contacts flagged as epileptogenic nodes."""
    recs = []
    for sub, g in df.groupby("subject"):
        res = g[g["region"] == "inside"]
        recs.append({"subject": sub, "outcome": g["outcome"].iloc[0],
                     "n_resected": len(res),
                     "overlap": float(res["is_node"].mean()) if len(res) else np.nan})
    return pd.DataFrame(recs)


def overlap_sweep(ov):
    """Sweep overlap thresholds; return the full table and the Youden-optimal row."""
    ov = ov.dropna(subset=["overlap"])
    good = (ov["outcome"] == "good").to_numpy()
    overlap = ov["overlap"].to_numpy()
    rows = []
    for t in OVERLAP_STEPS:
        above = overlap > t
        TP, FP = int(np.sum(above & good)), int(np.sum(above & ~good))
        TN, FN = int(np.sum(~above & ~good)), int(np.sum(~above & good))
        sens = TP / (TP + FN) if (TP + FN) else np.nan
        spec = TN / (TN + FP) if (TN + FP) else np.nan
        rows.append({
            "threshold": t, "TP": TP, "FP": FP, "TN": TN, "FN": FN,
            "PPV": TP / (TP + FP) if (TP + FP) else np.nan,
            "NPV": TN / (TN + FN) if (TN + FN) else np.nan,
            "accuracy": (TP + TN) / len(good),
            "youden": (sens + spec - 1) if not (np.isnan(sens) or np.isnan(spec)) else np.nan,
        })
    sweep = pd.DataFrame(rows)
    best = sweep.loc[sweep["youden"].idxmax()]
    return sweep, best


# Run the full procedure for the significant property, per band
overlap_results, sweeps = [], {}
for band in band_names:
    df_flag, p_thr, direction = flag_epileptogenic(OUTCOME_MEASURE, band)
    ov = patient_overlap(df_flag)
    sweep, best = overlap_sweep(ov)
    sweeps[band] = (sweep, best, ov)
    TP, FP, FN, TN = int(best.TP), int(best.FP), int(best.FN), int(best.TN)
    _, fisher_p = fisher_exact([[TP, FP], [FN, TN]])          # two-sided, as MATLAB fishertest.m
    overlap_results.append({
        "band": band, "sig_sec2": band in SIG_BANDS, "node_type": direction,
        "opt_overlap": best["threshold"], "PPV": best["PPV"], "NPV": best["NPV"],
        "accuracy": best["accuracy"], "fisher_p": fisher_p,
    })

overlap_df = pd.DataFrame(overlap_results)
pd.set_option("display.float_format", "{:.4f}".format)
print(f"Electrode-overlap outcome prediction using '{OUTCOME_MEASURE}':")
print(overlap_df.to_string(index=False))

In [ ]:
# ---- Figure: radar/spider plot of the overlap-prediction metrics per band ----
# One axis per frequency band; three series (PPV, NPV, accuracy) at each band's
# Youden-optimal overlap threshold. A "*" on a band label marks Fisher significance.
_metrics = [("PPV", "#C0453A", "s"), ("NPV", "#2C6E9E", "o"), ("accuracy", "#4B8B5E", "D")]
_labels = band_names
_ang = np.linspace(0, 2 * np.pi, len(_labels), endpoint=False)
_ang_c = np.concatenate([_ang, _ang[:1]])
_ov = overlap_df.set_index("band").reindex(_labels)
_fisher = _ov["fisher_p"]

with plt.rc_context({"font.size": 11, "figure.facecolor": "white", "axes.facecolor": "white"}):
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)          # Delta at the top
    ax.set_theta_direction(-1)              # clockwise

    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], color="0.55", fontsize=9)
    ax.set_rlabel_position(19)
    ax.grid(color="0.85", lw=0.9)
    ax.spines["polar"].set_color("0.85")
    ax.set_axisbelow(True)

    for _name, _colour, _marker in _metrics:
        _vals = np.nan_to_num(_ov[_name].to_numpy(float) * 100.0)
        _vc = np.concatenate([_vals, _vals[:1]])
        _lbl = "Accuracy" if _name == "accuracy" else _name.upper()
        ax.plot(_ang_c, _vc, color=_colour, lw=2.3, marker=_marker, markersize=8.5,
                markeredgecolor="white", markeredgewidth=1.3, label=_lbl, zorder=4, clip_on=False)
        ax.fill(_ang_c, _vc, color=_colour, alpha=0.11, zorder=2)

    ax.set_xticks(_ang)
    ax.set_xticklabels([f"{b.title()} *" if _fisher[b] < 0.05 else b.title() for b in _labels],
                       fontsize=11.5)
    ax.tick_params(axis="x", pad=12)
    for _t, b in zip(ax.get_xticklabels(), _labels):
        _t.set_fontweight("bold" if _fisher[b] < 0.05 else "normal")
        _t.set_color("0.12" if _fisher[b] < 0.05 else "0.4")

    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False,
              fontsize=11.5, handletextpad=0.4, columnspacing=1.8)
    ax.set_title("Outcome prediction by band \u2014 resected-node overlap metrics\n"
                 f"at the Youden-optimal overlap threshold  (Fisher's exact  *: p < 0.05, "
                 f"property = {OUTCOME_MEASURE})",
                 fontsize=13, fontweight="semibold", pad=26)
    fig.subplots_adjust(left=0.10, right=0.90, top=0.88, bottom=0.15)
    plt.show()


## 5. Where we end up

Read together, the four steps take the same two numbers per contact — **inward and outward directed
strength** — all the way from description to prediction:

- **Localisation (1-2).** Band by band, whether resected tissue differs from the rest of the brain in
  how it drives (outward) or receives (inward) directed influence, and whether that pattern is specific
  to good-outcome patients. If the pipeline reproduces (Stergiadis et al., 2026), the clearest signal is
  **lower outward strength inside the resection of good-outcome patients** at the higher bands, with no
  comparable effect in poor-outcome patients.
- **Classification (3).** Whether those two strengths are enough to *identify* resected (epileptogenic)
  contacts, summarised as a cross-validated AUC per band that is robust to the class imbalance.
- **Outcome prediction (4).** Whether the strength information can *predict* who becomes seizure-free —
  first with a compact contrast + leave-one-out AUC (4.1), then with the study's electrode-overlap
  procedure (4.2, Stergiadis et al., 2026): a Youden-optimal node threshold, a per-patient
  resected-hub/sink overlap, and a Fisher's exact test on the optimal overlap threshold, framed against
  prior work (Shah et al., 2019; Sinha et al., 2017; Corona et al., 2023).

Two things to keep in mind when comparing with the paper: this notebook uses **DTF** (the study used the
closely related **dDTF**), and it draws its segments **at random** rather than separating HFO-free data
(see notebook 2). The cohort is also small (n = 20), so the classification and outcome-prediction AUCs
are illustrative rather than definitive. The aim here is to show how such an analysis is built with
CobraBox, not to reproduce the paper exactly.